# 第10章：MCP（Model Context Protocol）

## 概念

**MCP** = 一种开放协议，让 AI 模型与外部工具/数据源标准化连接

```
┌──────────┐    MCP协议    ┌──────────────┐
│  LLM     │ ◄──────────► │  MCP Server  │
│  Client  │              │  (工具/资源)  │
└──────────┘              └──────────────┘
```

## 与 Tool Use 的区别

| | Tool Use | MCP |
|--|----------|-----|
| **定义** | LLM 调用工具的能力 | 工具连接的标准协议 |
| **类比** | 函数调用 | USB 接口标准 |
| **关系** | MCP 实现了 Tool Use | MCP 是协议层 |

## 代码演示

1. 创建 MCP Server（FastMCP）
2. 使用 MCP Client 连接
3. 集成 LangChain Agent

In [ ]:
import os
import asyncio
import nest_asyncio
from dotenv import load_dotenv
from fastmcp import FastMCP, Client
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [ ]:
nest_asyncio.apply()
load_dotenv()

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0,
)

print("MiMo 模型初始化完成！")

MiMo 模型初始化完成！


## Step 1：创建 MCP Server

使用 FastMCP 创建一个 MCP Server，暴露 3 个工具：
- `greet(name)` - 生成问候语
- `add(a, b)` - 两数相加
- `get_weather(city)` - 模拟查询天气

In [ ]:
# 创建 MCP Server
mcp_server = FastMCP("demo_server")


@mcp_server.tool()
def greet(name: str) -> str:
    """根据名字生成问候语"""
    return f"你好，{name}！欢迎使用 MCP 工具。"


@mcp_server.tool()
def add(a: int, b: int) -> str:
    """计算两个整数的和"""
    return f"{a} + {b} = {a + b}"


@mcp_server.tool()
def get_weather(city: str) -> str:
    """查询指定城市的天气（模拟数据）"""
    weather_data = {"北京": "晴天，25°C", "上海": "多云，22°C", "深圳": "阵雨，28°C", "杭州": "阴天，20°C"}
    return weather_data.get(city, f"{city}：暂无数据")


print("MCP Server 创建完成！")
print("已注册工具: greet, add, get_weather")

MCP Server 创建完成！
已注册工具: greet, add, get_weather


## Step 2：使用 MCP Client 连接

MCP Client 可以：
- `list_tools()` - 列出所有可用工具
- `call_tool(name, args)` - 调用指定工具

In [ ]:
async def demo_mcp_client():
    """演示 MCP Client 的使用"""
    async with Client(mcp_server) as client:
        # 1. 列出所有工具
        tools = await client.list_tools()
        print("=== MCP 工具列表 ===")
        for tool in tools:
            print(f"  - {tool.name}: {tool.description}")

        # 2. 调用工具
        print("\n=== 调用工具演示 ===")

        result = await client.call_tool("greet", {"name": "小明"})
        print(f"greet('小明') = {result.data}")

        result = await client.call_tool("add", {"a": 17, "b": 25})
        print(f"add(17, 25) = {result.data}")

        result = await client.call_tool("get_weather", {"city": "北京"})
        print(f"get_weather('北京') = {result.data}")


await demo_mcp_client()

=== MCP 工具列表 ===
  - greet: 根据名字生成问候语
  - add: 计算两个整数的和
  - get_weather: 查询指定城市的天气（模拟数据）

=== 调用工具演示 ===
greet('小明') = 你好，小明！欢迎使用 MCP 工具。
add(17, 25) = 17 + 25 = 42
get_weather('北京') = 晴天，25°C


## Step 3：集成 LangChain Agent

将 MCP 工具包装为 LangChain 工具，让 Agent 可以调用：

```
MiMo LLM → Agent → LangChain Tool → MCP Client → MCP Server
```

In [ ]:
# 同步调用 MCP 工具的包装器
_loop = asyncio.new_event_loop()


def call_mcp_tool(tool_name: str, **kwargs) -> str:
    """同步调用 MCP 工具"""

    async def _call():
        async with Client(mcp_server) as client:
            result = await client.call_tool(tool_name, kwargs)
            return result.data

    return _loop.run_until_complete(_call())


# 定义工具的输入 Schema
class GreetInput(BaseModel):
    name: str = Field(description="要问候的人名")


class AddInput(BaseModel):
    a: int = Field(description="第一个数")
    b: int = Field(description="第二个数")


class WeatherInput(BaseModel):
    city: str = Field(description="城市名称")


# 创建 LangChain 工具（包装 MCP 工具）
greet_tool = StructuredTool.from_function(
    func=lambda name: call_mcp_tool("greet", name=name),
    name="greet",
    description="根据名字生成问候语",
    args_schema=GreetInput,
)

add_tool = StructuredTool.from_function(
    func=lambda a, b: call_mcp_tool("add", a=a, b=b), name="add", description="计算两个整数的和", args_schema=AddInput
)

weather_tool = StructuredTool.from_function(
    func=lambda city: call_mcp_tool("get_weather", city=city),
    name="get_weather",
    description="查询指定城市的天气",
    args_schema=WeatherInput,
)

tools = [greet_tool, add_tool, weather_tool]
print("LangChain 工具创建完成！")
print(f"工具列表: {[t.name for t in tools]}")

LangChain 工具创建完成！
工具列表: ['greet', 'add', 'get_weather']


In [ ]:
# 创建 Agent 提示词
agent_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """你是一个助手，可以使用以下工具：
- greet: 根据名字生成问候语
- add: 计算两个整数的和
- get_weather: 查询城市天气

请根据用户问题选择合适的工具。""",
        ),
        ("user", "{question}"),
    ]
)


# 简单的工具调度函数
def agent_with_mcp(question: str):
    """Agent 调用 MCP 工具"""
    # 分析问题，选择工具
    analysis_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """分析用户问题，输出应该调用的工具和参数。

可用工具：
1. greet(name: str) - 问候
2. add(a: int, b: int) - 加法
3. get_weather(city: str) - 天气查询

输出格式（每行一个）：
工具名: xxx
参数: key=value,key=value

如果不需要工具，输出：工具名: 无""",
            ),
            ("user", "{question}"),
        ]
    )

    chain = analysis_prompt | llm | StrOutputParser()
    analysis = chain.invoke({"question": question})
    print(f"分析结果: {analysis}")

    # 解析并调用工具
    lines = analysis.strip().split("\n")
    tool_name = ""
    params = {}

    for line in lines:
        if line.startswith("工具名:"):
            tool_name = line.split(":")[1].strip()
        elif line.startswith("参数:"):
            param_str = line.split(":")[1].strip()
            for p in param_str.split(","):
                if "=" in p:
                    k, v = p.split("=")
                    # 尝试转为整数
                    try:
                        params[k.strip()] = int(v.strip())
                    except ValueError:
                        params[k.strip()] = v.strip()

    if tool_name == "无" or not tool_name:
        return "无需调用工具。"

    # 调用 MCP 工具
    result = call_mcp_tool(tool_name, **params)
    return result


# 测试
print("=== 测试 1：问候 ===")
print(agent_with_mcp("帮我问候一下小红"))

print("\n=== 测试 2：加法 ===")
print(agent_with_mcp("计算 123 + 456"))

print("\n=== 测试 3：天气 ===")
print(agent_with_mcp("北京今天天气怎么样？"))

=== 测试 1：问候 ===
分析结果: 工具名: greet
参数: name=小红
你好，小红！欢迎使用 MCP 工具。

=== 测试 2：加法 ===
分析结果: 工具名: add
参数: a=123,b=456
123 + 456 = 579

=== 测试 3：天气 ===
分析结果: 工具名: get_weather
参数: city=北京
晴天，25°C


## MCP 架构总结

```
┌─────────────────────────────────────────────────────────┐
│                    MCP 架构                              │
├─────────────────────────────────────────────────────────┤
│  MiMo LLM                                               │
│     ↓                                                   │
│  LangChain Agent                                        │
│     ↓                                                   │
│  LangChain Tools（包装 MCP Client）                      │
│     ↓                                                   │
│  MCP Client  ←── MCP 协议 ──→  MCP Server              │
│  (fastmcp)                      (FastMCP)               │
│                                 ↓                       │
│                            工具实现                      │
│                         (greet/add/weather)             │
└─────────────────────────────────────────────────────────┘
```

## MCP 核心概念

| 概念 | 说明 | 示例 |
|------|------|------|
| **MCP Server** | 暴露工具/资源的服务 | FastMCP Server |
| **MCP Client** | 连接 Server 的客户端 | `Client(mcp_server)` |
| **Tool** | 可调用的函数 | `greet`, `add`, `get_weather` |
| **Transport** | 通信方式 | stdio / HTTP / 内存 |

## MCP vs 直接 Tool Use

| | 直接 Tool Use | MCP |
|--|--------------|-----|
| **定义方式** | Python 函数 | MCP Server 暴露 |
| **发现** | 代码中硬编码 | Client 动态发现 |
| **跨语言** | 需要适配 | 协议统一 |
| **适用场景** | 单项目 | 多工具/多项目复用 |

## 运行方式

```bash
pip install fastmcp
cd chapter_notebooks/chapter_10_mcp
jupyter notebook ch10_mcp_langchain.ipynb
```